In [2]:

import sys
sys.path.insert(0, "/home/xilinx/jupyter_notebooks/qick/qick_lib")
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from scipy import signal as scipy_signal
from qick import *

# -----------------------------------------------------------------------------
# 1. HARDWARE & INITIALIZATION SETUP
# -----------------------------------------------------------------------------
soc = QickSoc()
soccfg = soc

GEN_CH = 1
RO_CH = 0

gencfg = soccfg["gens"][GEN_CH]
samps_per_clk = gencfg["samps_per_clk"]
ENV_SR = gencfg["f_fabric"] * samps_per_clk * 1e6
ENV_MAXLEN = gencfg["maxlen"]

print(f"Generator {GEN_CH}: f_fabric={gencfg['f_fabric']:.3f} MHz, "
      f"samps_per_clk={samps_per_clk}, envelope sample rate={ENV_SR/1e9:.4f} GSPS")
print(f"Envelope memory available: {ENV_MAXLEN} samples")

# -----------------------------------------------------------------------------
# 2. SINGLE-TONE SERRODYNE STEP GENERATOR (continuous phase across steps)
#    FIX #3: n_samples is chosen from a small grid-aligned neighborhood
#    around the target duration, picking whichever candidate leaves the
#    smallest residual phase at the periodic-buffer wraparound point (i.e.
#    closest to landing on a whole number of cycles). This suppresses the
#    sidebands caused by mode="periodic" re-triggering on a non-continuous
#    phase boundary.
# -----------------------------------------------------------------------------
def best_n_samples(freq_hz, target_dur_s, sample_rate, samps_per_clk_, search_radius=3):
    target_n = int(round(target_dur_s * sample_rate))
    target_n = max(target_n, 1)
    # snap up to the fabric-cycle grid first
    pad0 = (-target_n) % samps_per_clk_
    target_n += pad0

    candidates = [target_n + k * samps_per_clk_
                  for k in range(-search_radius, search_radius + 1)]
    candidates = [n for n in candidates if n >= samps_per_clk_]

    if freq_hz == 0 or len(candidates) == 0:
        return target_n

    def residual(n):
        cycles = freq_hz * n / sample_rate
        frac = cycles - np.floor(cycles)
        return min(frac, 1.0 - frac)  # distance to nearest integer number of cycles

    # Tie-break on distance from target_n, not list order: when freq_hz is
    # near zero, many candidates have near-identical (near-zero) residual,
    # and picking the first such candidate can land arbitrarily far from
    # target_n -- causing large, unnecessary step-to-step buffer-length
    # variance once ratios/amplitudes differ per step. Rounding damps
    # floating-point noise so true near-ties are broken by proximity.
    return min(candidates, key=lambda n: (round(residual(n), 9), abs(n - target_n)))


def serrodyne_tone(freq_hz, duration_sec, sample_rate, amplitude, phase0=0.0, width=1.0,
                    n_samples_override=None):
    if n_samples_override is not None:
        n_samples = max(1, int(n_samples_override))
    else:
        n_samples = max(1, int(round(float(duration_sec) * sample_rate)))
    dt = 1.0 / sample_rate
    t = np.arange(n_samples) * dt
    phase = 2 * np.pi * freq_hz * t + phase0
    y = float(amplitude) * scipy_signal.sawtooth(phase, width=width)
    phase_end = (phase0 + 2 * np.pi * freq_hz * n_samples * dt) % (2 * np.pi)
    return y, phase_end, n_samples

# -----------------------------------------------------------------------------
# 3. CHIRP DEFINITION -- each step's buffer is a 4-tone composite.
#    All NUM_TONES tones in a given step share the same chirp_offset_hz for
#    that step (the sweep frequency), but each step now gets its own
#    ratio/amplitude row -- see the PER-STEP ARRAYS block below.
#
#    NOTE: the sweep endpoint (CHIRP_OFFSET_STOP_HZ) and the trap frequency
#    (TRAP_OFFSET_HZ) are different values -- the sweep goes from
#    -230 MHz up to +120 MHz, then the trap buffer is built separately at
#    0 MHz. Phase is carried continuously from the last sweep step into the
#    trap buffer, but since the trap frequency != sweep endpoint, there is
#    an unavoidable, instantaneous ~120 MHz frequency jump at the
#    sweep-to-trap handoff (phase-continuous, but not frequency-continuous).
# -----------------------------------------------------------------------------
NUM_STEPS = 45                       # number of sweep steps -- just tune this directly
TOTAL_BUFFERS = NUM_STEPS + 1        # + 1 trap buffer = 45 total composite buffers

CHIRP_OFFSET_START_HZ = -230e6       # sweep start
CHIRP_OFFSET_STOP_HZ = 120e6         # sweep end
TRAP_OFFSET_HZ = 0.0                 # frequency to hold at after the sweep

BASE_TONES_HZ = np.array([76.25e6, 0, -122.92e6+76.25e6, -147.82e6+76.25e6])
BASE_TONES_HZ *= -1
NUM_TONES = len(BASE_TONES_HZ)

# -----------------------------------------------------------------------------
# PER-STEP, PER-TONE RATIOS AND AMPLITUDES
#
# Both are (TOTAL_BUFFERS, NUM_TONES) = (45, 4) arrays: one row per
# composite buffer -- rows 0..NUM_STEPS-1 are the 44 sweep steps (in the
# same order as the chirp sweeps from CHIRP_OFFSET_START_HZ to
# CHIRP_OFFSET_STOP_HZ), and row NUM_STEPS (the 45th row) is the trap
# buffer. Columns follow BASE_TONES_HZ's tone order.
#
#   TONE_RATIOS_PER_STEP[i, j]  -- relative duration weight of tone j within
#                                   step i's composite buffer. Only the
#                                   ratio between the 4 numbers in a row
#                                   matters; each row is re-normalized to
#                                   sum to 1 independently, so rows don't
#                                   need to sum to the same total.
#   AMPLITUDE_PER_STEP[i, j]    -- serrodyne amplitude (0-1) of tone j
#                                   within step i.
#
# Defaults below just replay the old fixed global values on every
# step/tone -- edit these two arrays directly (or build them
# programmatically, e.g. by interpolating between two rows) to vary
# ratios/amplitudes across the sweep and at the trap.
# -----------------------------------------------------------------------------
DEFAULT_TONE_RATIOS = np.array([0.337, 0.167, 0.288, 0.208])
DEFAULT_AMPLITUDE = 0.5

TONE_RATIOS_PER_STEP = np.tile(DEFAULT_TONE_RATIOS, (TOTAL_BUFFERS, 1))   # (45, 4)
AMPLITUDE_PER_STEP = np.full((TOTAL_BUFFERS, NUM_TONES), DEFAULT_AMPLITUDE)  # (45, 4)

AMPLITUDE_PER_STEP = [[1.394, 1.587, 1.705, 1.768],
 [1.374, 1.567, 1.685, 1.748],
 [1.353, 1.547, 1.665, 1.728],
 [1.333, 1.526, 1.645, 1.708],
 [1.313, 1.506, 1.625, 1.688],
 [1.293, 1.486, 1.604, 1.668],
 [1.273, 1.466, 1.584, 1.647],
 [1.253, 1.446, 1.564, 1.627],
 [1.232, 1.426, 1.544, 1.607],
 [1.212, 1.406, 1.524, 1.587],
 [1.192, 1.385, 1.504, 1.567],
 [1.172, 1.365, 1.483, 1.547],
 [1.152, 1.345, 1.463, 1.526],
 [1.132, 1.325, 1.443, 1.506],
 [1.089, 1.305, 1.423, 1.486],
 [1.079, 1.285, 1.403, 1.466],
 [1.031, 1.264, 1.383, 1.446],
 [1.007, 1.244, 1.363, 1.426],
 [1.014, 1.224, 1.342, 1.405],
 [1.01 , 1.204, 1.322, 1.385],
 [1.01 , 1.184, 1.302, 1.365],
 [1.01 , 1.164, 1.282, 1.345],
 [1.01 , 1.143, 1.262, 1.325],
 [1.01 , 1.085, 1.242, 1.305],
 [1.01 , 1.087, 1.221, 1.285],
 [1.01 , 1.07 , 1.201, 1.264],
 [1.01 , 1.004, 1.181, 1.244],
 [1.01 , 1.013, 1.161, 1.224],
 [1.01 , 1.012, 1.141, 1.204],
 [1.01 , 1.01 , 1.086, 1.184],
 [1.01 , 1.01 , 1.085, 1.164],
 [1.01 , 1.01 , 1.061, 1.143],
 [1.01 , 1.01 , 1.001, 1.085],
 [1.01 , 1.01 , 1.015, 1.087],
 [1.01 , 1.01 , 1.011, 1.069],
 [1.01 , 1.01 , 1.01 , 1.004],
 [1.01 , 1.01 , 1.01 , 1.013],
 [1.01 , 1.01 , 1.01 , 1.012],
 [1.01 , 1.01 , 1.01 , 1.01 ],
 [1.01 , 1.01 , 1.01 , 1.01 ],
 [1.01 , 1.01 , 1.01 , 1.01 ],
 [1.01 , 1.01 , 1.01 , 1.01 ],
 [1.01 , 1.01 , 1.01 , 1.01 ],
 [1.01 , 1.01 , 1.01 , 1.01 ],
 [1.01 , 1.01 , 1.01 , 1.01 ]]

assert TONE_RATIOS_PER_STEP.shape == (TOTAL_BUFFERS, NUM_TONES), (
    f"TONE_RATIOS_PER_STEP must be shape ({TOTAL_BUFFERS}, {NUM_TONES}), "
    f"got {TONE_RATIOS_PER_STEP.shape}"
)
assert AMPLITUDE_PER_STEP.shape == (TOTAL_BUFFERS, NUM_TONES), (
    f"AMPLITUDE_PER_STEP must be shape ({TOTAL_BUFFERS}, {NUM_TONES}), "
    f"got {AMPLITUDE_PER_STEP.shape}"
)
assert np.all(TONE_RATIOS_PER_STEP.sum(axis=1) > 0), "Each row of TONE_RATIOS_PER_STEP must sum to > 0"
assert np.all(AMPLITUDE_PER_STEP >= 0) and np.all(AMPLITUDE_PER_STEP <= 1), (
    "AMPLITUDE_PER_STEP entries must be within [0, 1]"
)

TOTAL_SWEEP_S = 6e-3  # 6e-3 -- this is the PHYSICAL sweep duration (NUM_STEPS x
                       # STEP_HOLD_US), fixed by TOTAL_SWEEP_S / NUM_STEPS below.
                       # It is independent of CYCLE_S: CYCLE_S only sets how many
                       # times each step's composite buffer loops (mode="periodic")
                       # within its STEP_HOLD_US dwell -- so CYCLE_S can be shrunk
                       # automatically (see the memory-fit loop below) without
                       # changing the sweep's physical timing.
max_feasible_total_s = 0.95 * ENV_MAXLEN / ENV_SR
CYCLE_S = max_feasible_total_s / TOTAL_BUFFERS   # initial (nominal) composite-buffer duration

tone_fractions_per_step = TONE_RATIOS_PER_STEP / TONE_RATIOS_PER_STEP.sum(axis=1, keepdims=True)

STEP_HOLD_S = TOTAL_SWEEP_S / NUM_STEPS


STEP_HOLD_US = STEP_HOLD_S * 1e6

print(f"Nominal composite buffer cycle: {CYCLE_S*1e9:.2f} ns (may shrink to fit memory -- "
      f"see below), held via mode='periodic' for {STEP_HOLD_US:.2f} us per chirp step "
      f"({TOTAL_SWEEP_S*1e3:.3f} ms total sweep)")
print("Per-step tone ratios/amplitudes (step 0 shown as example, at the nominal cycle; "
      "edit TONE_RATIOS_PER_STEP / AMPLITUDE_PER_STEP to vary per step):")
_preview_durations = tone_fractions_per_step[0] * CYCLE_S
for j in range(NUM_TONES):
    print(f"  tone {j} ({BASE_TONES_HZ[j]/1e6:+.1f} MHz offset): "
          f"step0 ratio {TONE_RATIOS_PER_STEP[0, j]:.3f}, amp {AMPLITUDE_PER_STEP[0, j]:.3f} "
          f"-> {_preview_durations[j]*1e9:.2f} ns")

def build_multitone_buffer(chirp_offset_hz, phase0, tone_durations_s_row, amplitudes_row):
    """One composite envelope: all NUM_TONES tones, each shifted by the same
    chirp_offset_hz, using this buffer's own per-tone durations and
    amplitudes (a single row of tone_durations_per_step_s / AMPLITUDE_PER_STEP),
    concatenated. Sample count per tone is chosen (within a small
    grid-aligned window) to minimize the phase residual at the
    mode="periodic" wraparound point."""
    i_pieces, q_pieces = [], []
    phase = phase0
    for base_tone, dur, amp in zip(BASE_TONES_HZ, tone_durations_s_row, amplitudes_row):
        f = chirp_offset_hz + base_tone
        n_best = best_n_samples(f, dur, ENV_SR, samps_per_clk)
        y, phase, n_samples = serrodyne_tone(f, dur, ENV_SR, amplitude=amp, phase0=phase,
                                              n_samples_override=n_best)
        pad = (-len(y)) % samps_per_clk
        if pad:
            y = np.concatenate([y, np.zeros(pad)])
        i_pieces.append(np.round(y * maxv).astype(np.int16))
        q_pieces.append(np.zeros(len(y), dtype=np.int16))
    return np.concatenate(i_pieces), np.concatenate(q_pieces), phase, [len(p) for p in i_pieces]

# -----------------------------------------------------------------------------
# FIX #1: midpoint chirp sampling instead of edge sampling.
# Each held step now represents the chirp's average frequency over that
# dwell interval (evaluated at the interval midpoint), rather than the
# frequency at the interval's start. This makes the instantaneous phase
# error within a step symmetric (+-beta*tau^2/8) instead of one-sided
# (0 to beta*tau^2/2) -- a 4x reduction in peak phase error for free.
# The very last step is snapped exactly to CHIRP_OFFSET_STOP_HZ (the sweep
# endpoint, +120 MHz) so the sweep itself stays phase-continuous right up
# to its final step. The trap buffer (0 MHz) is built separately below.
# -----------------------------------------------------------------------------
step_width_hz = (CHIRP_OFFSET_STOP_HZ - CHIRP_OFFSET_START_HZ) / NUM_STEPS
chirp_offsets_hz = CHIRP_OFFSET_START_HZ + (np.arange(NUM_STEPS) + 0.5) * step_width_hz
chirp_offsets_hz[-1] = CHIRP_OFFSET_STOP_HZ  # land exactly on sweep end (+120 MHz)

maxv = soccfg.get_maxv(GEN_CH)

def build_all_buffers(cycle_s):
    """Build every sweep-step buffer plus the trap buffer at composite-buffer
    duration cycle_s, using each row's own ratios/amplitudes. Returns the raw
    (pre-equalization) sweep buffers, the trap buffer, and the flat list of
    every individual tone-slice length (for the hardware floor check)."""
    durations_s = tone_fractions_per_step * cycle_s  # (45, 4)

    idata_raw, qdata_raw = [], []
    all_piece_lens = []
    phase = 0.0
    for i, offset in enumerate(chirp_offsets_hz):
        idata, qdata, phase, piece_lens = build_multitone_buffer(
            offset, phase, durations_s[i], AMPLITUDE_PER_STEP[i]
        )
        idata_raw.append(idata)
        qdata_raw.append(qdata)
        all_piece_lens.extend(piece_lens)

    trap_idata_, trap_qdata_, _, trap_piece_lens_ = build_multitone_buffer(
        TRAP_OFFSET_HZ, phase, durations_s[NUM_STEPS], AMPLITUDE_PER_STEP[NUM_STEPS]
    )
    all_piece_lens.extend(trap_piece_lens_)
    return idata_raw, qdata_raw, trap_idata_, trap_qdata_, trap_piece_lens_, all_piece_lens

# -----------------------------------------------------------------------------
# MEMORY-FIT LOOP WITH AUTOMATIC BUFFER-LENGTH EQUALIZATION
#
# The tProc addresses consecutive "serr_i" envelope buffers by incrementing
# a fixed address step each time the sweep advances -- that only works if
# every sweep-step buffer occupies the same number of envelope samples. Since
# per-tone durations are individually snapped to the fabric-clock grid
# (best_n_samples) and per-step ratios can differ row to row, raw buffer
# lengths can vary step to step; they're padded up to the longest one so all
# NUM_STEPS buffers come out equal length.
#
# That padding can push the total over ENV_MAXLEN. Since CYCLE_S (how many
# times each step's buffer loops within its fixed STEP_HOLD_US dwell) has no
# effect on the sweep's physical timing, it's safe to shrink automatically
# and rebuild until sweep + trap fit in memory, rather than requiring manual
# retuning of NUM_STEPS or the ratio/amplitude arrays.
# -----------------------------------------------------------------------------
SHRINK_FACTOR = 0.98
MAX_FIT_ATTEMPTS = 60

cycle_s = CYCLE_S
for attempt in range(1, MAX_FIT_ATTEMPTS + 1):
    idata_list_raw, qdata_list_raw, trap_idata, trap_qdata, trap_piece_lens, all_piece_lens = \
        build_all_buffers(cycle_s)

    min_piece_len = min(all_piece_lens)
    if min_piece_len < 3 * samps_per_clk:
        raise ValueError(
            f"Smallest tone slice is only {min_piece_len} samples "
            f"({min_piece_len/samps_per_clk:.1f} fabric cycles) at cycle_s={cycle_s*1e9:.2f} ns "
            f"(attempt {attempt}) -- hardware needs >= 3. Shrinking the buffer further only makes "
            f"this worse, so this must be fixed by hand: adjust TONE_RATIOS_PER_STEP (avoid "
            f"near-zero ratios in any row), or reduce NUM_STEPS."
        )

    raw_lengths = [len(x) for x in idata_list_raw]
    samples_per_step = max(raw_lengths)
    samples_per_step += (-samples_per_step) % samps_per_clk  # round up to fabric-cycle grid
    total_samples = samples_per_step * NUM_STEPS
    total_with_trap = total_samples + len(trap_idata)

    if total_with_trap <= ENV_MAXLEN:
        break
    cycle_s *= SHRINK_FACTOR
else:
    raise ValueError(
        f"Could not fit sweep + trap buffers in {ENV_MAXLEN} samples after "
        f"{MAX_FIT_ATTEMPTS} shrink attempts. Reduce NUM_STEPS, or make "
        f"TONE_RATIOS_PER_STEP less extreme (less step-to-step variation)."
    )

if attempt > 1:
    print(f"Composite buffer cycle shrunk from {CYCLE_S*1e9:.2f} ns to {cycle_s*1e9:.2f} ns "
          f"over {attempt} attempts to fit envelope memory (sweep timing unaffected).")

print(f"Smallest individual tone slice across all steps (incl. trap): {min_piece_len} samples "
      f"({min_piece_len/samps_per_clk:.1f} fabric cycles)")
print(f"Raw per-step buffer lengths (pre-padding): {sorted(set(raw_lengths))}")

idata_list, qdata_list = [], []
for idata, qdata in zip(idata_list_raw, qdata_list_raw):
    pad = samples_per_step - len(idata)
    if pad:
        idata = np.concatenate([idata, np.zeros(pad, dtype=np.int16)])
        qdata = np.concatenate([qdata, np.zeros(pad, dtype=np.int16)])
    idata_list.append(idata)
    qdata_list.append(qdata)

assert all(len(x) == samples_per_step for x in idata_list), "Sweep buffers not equal length after padding"
assert samples_per_step % samps_per_clk == 0

print(f"Equalized per-step buffer length: {samples_per_step} samples "
      f"(shortest raw step was {min(raw_lengths)} samples, so up to "
      f"{samples_per_step - min(raw_lengths)} samples of trailing zero padding added)")
print(f"Total envelope samples (sweep): {total_samples} / {ENV_MAXLEN} available")
print(f"Trap buffer: {len(trap_idata)} samples (tone slices: {trap_piece_lens})")
print(f"Total envelope samples (sweep + trap): {total_with_trap} / {ENV_MAXLEN} available")

# -----------------------------------------------------------------------------
# 4. PROGRAM: sweep as before, then ONE final pulse on the concatenated trap
#    buffer with mode="periodic" -- hardware loops it forever on its own.
# -----------------------------------------------------------------------------
class RepeatedStepSerrodyneProgram(RAveragerProgram):
    def initialize(self):
        cfg = self.cfg
        res_ch = cfg["res_ch"]

        self.declare_gen(ch=res_ch, nqz=1)

        for i, (idata_step, qdata_step) in enumerate(zip(cfg["idata_list"], cfg["qdata_list"])):
            self.add_envelope(ch=res_ch, name=f"serr_{i}", idata=idata_step, qdata=qdata_step)

        self.add_envelope(ch=res_ch, name="trap_wfm", idata=cfg["trap_idata"], qdata=cfg["trap_qdata"])

        self.set_pulse_registers(
            ch=res_ch,
            style="arb",
            freq=0,
            phase=0,
            gain=cfg["gain"],
            waveform="serr_0",
            outsel="input",
            mode="periodic",
        )

        self.r_rp = self.ch_page(res_ch)
        self.r_addr = self.sreg(res_ch, "addr")
        self.addr_step = cfg["samples_per_step"] // self.soccfg["gens"][res_ch]["samps_per_clk"]

        self.synci(200)

    def body(self):
        res_ch = self.cfg["res_ch"]
        step_cycles = self.us2cycles(self.cfg["step_hold_us"])

        self.trigger(pins=[0])
        self.pulse(ch=res_ch, t='auto')
        self.sync_all(step_cycles)

    def update(self):
        self.mathi(self.r_rp, self.r_addr, self.r_addr, '+', self.addr_step)

    def make_program(self):
        """Standard RAveragerProgram sweep (expts x reps), with a single
        trapping pulse appended after the loop -- no loop construct needed
        for trapping, since mode="periodic" makes the hardware repeat it
        on its own once triggered."""
        p = self
        rcount = 13
        rii = 14
        rjj = 15

        p.initialize()
        p.regwi(0, rcount, 0)
        p.regwi(0, rii, self.cfg['expts'] - 1)
        p.label("LOOP_I")
        p.regwi(0, rjj, self.cfg['reps'] - 1)
        p.label("LOOP_J")
        p.body()
        p.mathi(0, rcount, rcount, "+", 1)
        p.memwi(0, rcount, self.COUNTER_ADDR)
        p.loopnz(0, rjj, 'LOOP_J')
        p.update()
        p.loopnz(0, rii, "LOOP_I")

        # --- trapping: one pulse, periodic buffer, then end(). The DAC keeps
        # cycling the 4 concatenated tones forever regardless of what the
        # tProc does after this -- including after it hits end() and halts.
        res_ch = self.cfg["res_ch"]
        p.set_pulse_registers(
            ch=res_ch, style="arb", freq=0, phase=0, gain=self.cfg["gain"],
            waveform="trap_wfm", outsel="input", mode="periodic",
        )
        p.trigger(pins=[0])
        p.pulse(ch=res_ch, t='auto')
        p.end()

# -----------------------------------------------------------------------------
# 5. EXECUTION
# -----------------------------------------------------------------------------
config = {
    "res_ch": GEN_CH,
    "reps": 1,
    "expts": NUM_STEPS,
    "idata_list": idata_list,
    "qdata_list": qdata_list,
    "trap_idata": trap_idata,
    "trap_qdata": trap_qdata,
    "samples_per_step": samples_per_step,
    "step_hold_us": STEP_HOLD_US,
    "gain": 32767,
}

prog = RepeatedStepSerrodyneProgram(soccfg, config)
# prog.run(soc) #, start_src="external"
prog.run(soc, start_src="external")
print(f"Running on hardware — {NUM_STEPS} sweep steps x {STEP_HOLD_US:.2f} us "
      f"= {NUM_STEPS*STEP_HOLD_US*1e-3:.3f} ms sweep ({CHIRP_OFFSET_START_HZ/1e6:.0f} -> "
      f"{CHIRP_OFFSET_STOP_HZ/1e6:.0f} MHz), then trapping at {TRAP_OFFSET_HZ/1e6:.0f} MHz "
      f"(4 tones, periodic, indefinitely until soc.reset_gens()).")

print(TONE_RATIOS_PER_STEP)
print(AMPLITUDE_PER_STEP)

/usr/local/share/pynq-venv/lib/python3.10/site-packages/pydantic/_internal/_config.py:386: UserWarning: Valid config keys have changed in V2:
* 'underscore_attrs_are_private' has been removed
  warnings.warn(message, UserWarning)
/usr/local/share/pynq-venv/lib/python3.10/site-packages/pydantic/_internal/_config.py:386: UserWarning: Valid config keys have changed in V2:
* 'underscore_attrs_are_private' has been removed
  warnings.warn(message, UserWarning)


Generator 1: f_fabric=614.400 MHz, samps_per_clk=16, envelope sample rate=9.8304 GSPS
Envelope memory available: 65536 samples


AttributeError: 'list' object has no attribute 'shape'

In [4]:
# above sucks so claude did below

In [18]:
import sys
sys.path.insert(0, "/home/xilinx/jupyter_notebooks/qick/qick_lib")
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from scipy import signal as scipy_signal
from scipy import optimize
from qick import *

# -----------------------------------------------------------------------------
# 1. HARDWARE & INITIALIZATION SETUP
# -----------------------------------------------------------------------------
soc = QickSoc()
soccfg = soc

GEN_CH = 0
RO_CH = 0

gencfg = soccfg["gens"][GEN_CH]
samps_per_clk = gencfg["samps_per_clk"]
ENV_SR = gencfg["f_fabric"] * samps_per_clk * 1e6
ENV_MAXLEN = gencfg["maxlen"]

print(f"Generator {GEN_CH}: f_fabric={gencfg['f_fabric']:.3f} MHz, "
      f"samps_per_clk={samps_per_clk}, envelope sample rate={ENV_SR/1e9:.4f} GSPS")
print(f"Envelope memory available: {ENV_MAXLEN} samples")

# -----------------------------------------------------------------------------
# 2. SINGLE-TONE SERRODYNE STEP GENERATOR (continuous phase across steps)
#    FIX #3: n_samples is chosen from a small grid-aligned neighborhood
#    around the target duration, picking whichever candidate leaves the
#    smallest residual phase at the periodic-buffer wraparound point (i.e.
#    closest to landing on a whole number of cycles). This suppresses the
#    sidebands caused by mode="periodic" re-triggering on a non-continuous
#    phase boundary.
# -----------------------------------------------------------------------------
def best_n_samples(freq_hz, target_dur_s, sample_rate, samps_per_clk_, search_radius=3):
    target_n = int(round(target_dur_s * sample_rate))
    target_n = max(target_n, 1)
    # snap up to the fabric-cycle grid first
    pad0 = (-target_n) % samps_per_clk_
    target_n += pad0

    candidates = [target_n + k * samps_per_clk_
                  for k in range(-search_radius, search_radius + 1)]
    candidates = [n for n in candidates if n >= samps_per_clk_]

    if freq_hz == 0 or len(candidates) == 0:
        return target_n

    def residual(n):
        cycles = freq_hz * n / sample_rate
        frac = cycles - np.floor(cycles)
        return min(frac, 1.0 - frac)  # distance to nearest integer number of cycles

    # Tie-break on distance from target_n, not list order: when freq_hz is
    # near zero, many candidates have near-identical (near-zero) residual,
    # and picking the first such candidate can land arbitrarily far from
    # target_n -- causing large, unnecessary step-to-step buffer-length
    # variance once ratios/amplitudes differ per step. Rounding damps
    # floating-point noise so true near-ties are broken by proximity.
    return min(candidates, key=lambda n: (round(residual(n), 9), abs(n - target_n)))


def serrodyne_tone(freq_hz, duration_sec, sample_rate, amplitude, phase0=0.0, width=1.0,
                    n_samples_override=None):
    if n_samples_override is not None:
        n_samples = max(1, int(n_samples_override))
    else:
        n_samples = max(1, int(round(float(duration_sec) * sample_rate)))
    dt = 1.0 / sample_rate
    t = np.arange(n_samples) * dt
    phase = 2 * np.pi * freq_hz * t + phase0
    y = float(amplitude) * scipy_signal.sawtooth(phase, width=width)
    phase_end = (phase0 + 2 * np.pi * freq_hz * n_samples * dt) % (2 * np.pi)
    return y, phase_end, n_samples

# -----------------------------------------------------------------------------
# 3. CHIRP DEFINITION -- each step's buffer is a 4-tone composite.
#    All NUM_TONES tones in a given step share the same chirp_offset_hz for
#    that step (the sweep frequency), but each step now gets its own
#    ratio/amplitude row -- see the PER-STEP ARRAYS block below.
#
#    NOTE: the sweep endpoint (CHIRP_OFFSET_STOP_HZ) and the trap frequency
#    (TRAP_OFFSET_HZ) are different values -- the sweep goes from
#    -230 MHz up to +120 MHz, then the trap buffer is built separately at
#    0 MHz. Phase is carried continuously from the last sweep step into the
#    trap buffer, but since the trap frequency != sweep endpoint, there is
#    an unavoidable, instantaneous ~120 MHz frequency jump at the
#    sweep-to-trap handoff (phase-continuous, but not frequency-continuous).
# -----------------------------------------------------------------------------
NUM_STEPS = 45                       # number of sweep steps -- just tune this directly
TOTAL_BUFFERS = NUM_STEPS + 1        # + 1 trap buffer

CHIRP_OFFSET_START_HZ = -230e6       # sweep start
CHIRP_OFFSET_STOP_HZ = 120e6         # sweep end
TRAP_OFFSET_HZ = 0.0                 # frequency to hold at after the sweep

BASE_TONES_HZ = np.array([76.25e6, 0, -122.92e6+76.25e6, -147.82e6+76.25e6])
BASE_TONES_HZ *= -1
NUM_TONES = len(BASE_TONES_HZ)

# -----------------------------------------------------------------------------
# FIX #1: midpoint chirp sampling instead of edge sampling (computed here,
# early, because the amplitude calibration below needs the actual per-step
# sweep frequencies).
# Each held step represents the chirp's average frequency over that dwell
# interval (evaluated at the interval midpoint), rather than the frequency
# at the interval's start. This makes the instantaneous phase error within
# a step symmetric (+-beta*tau^2/8) instead of one-sided (0 to
# beta*tau^2/2) -- a 4x reduction in peak phase error for free. The very
# last step is snapped exactly to CHIRP_OFFSET_STOP_HZ (the sweep endpoint,
# +120 MHz) so the sweep itself stays phase-continuous right up to its
# final step. The trap buffer (0 MHz) is a separate, 46th frequency.
# -----------------------------------------------------------------------------
step_width_hz = (CHIRP_OFFSET_STOP_HZ - CHIRP_OFFSET_START_HZ) / NUM_STEPS
chirp_offsets_hz = CHIRP_OFFSET_START_HZ + (np.arange(NUM_STEPS) + 0.5) * step_width_hz
chirp_offsets_hz[-1] = CHIRP_OFFSET_STOP_HZ  # land exactly on sweep end (+120 MHz)

# -----------------------------------------------------------------------------
# AMPLITUDE CALIBRATION -- diffraction-efficiency correction vs. frequency
#
# amps/freq below are the measured diffraction efficiency at each tested
# frequency (MHz). We keep only the freq < 0 MHz side, normalize so its max
# is 1, then define correction_multiplier = 1 / normalized_amplitude: the
# factor you multiply a nominal drive amplitude by to compensate for lower
# efficiency at that frequency. Below -50 MHz (sparser, noisier data) we
# extrapolate with a linear fit instead of interpolating the raw points.
# -----------------------------------------------------------------------------
_CAL_AMPS = np.array([0.3933, 0.4029, 0.4029, 0.4365, 0.4077, 0.4149, 0.4532, 0.458 , 0.458 ,
 0.458 , 0.4604, 0.4053, 0.4604, 0.5012, 0.5012, 0.4772, 0.5108, 0.47  ,
 0.4748, 0.5276, 0.5564, 0.5468, 0.5156, 0.5252, 0.6139, 0.5659, 0.6091,
 0.6115, 0.5947, 0.6187, 0.6211, 0.6427, 0.6667, 0.6978, 0.693 , 0.7026,
 0.7554, 0.7434, 0.7482, 0.8417, 0.8417, 0.8537, 0.9017, 0.9257, 1.    ])

_CAL_FREQ_MHZ = np.array([-350.        ,-340.90909091,-331.81818182,-322.72727273,-313.63636364,
 -304.54545455,-295.45454545,-286.36363636,-277.27272727,-268.18181818,
 -259.09090909,-250.        ,-240.90909091,-231.81818182,-222.72727273,
 -213.63636364,-204.54545455,-195.45454545,-186.36363636,-177.27272727,
 -168.18181818,-159.09090909,-150.        ,-140.90909091,-131.81818182,
 -122.72727273,-113.63636364,-104.54545455, -95.45454545, -86.36363636,
  -77.27272727, -68.18181818, -59.09090909, -50.        , -40.90909091,
  -31.81818182, -22.72727273, -13.63636364,  -4.54545455,   4.54545455,
   13.63636364,  22.72727273,  31.81818182,  40.90909091,  50.        ])

_cal_mask = _CAL_FREQ_MHZ < 0
_cal_freq_mhz = _CAL_FREQ_MHZ[_cal_mask]
_cal_amps = _CAL_AMPS[_cal_mask]
_cal_amps = _cal_amps / np.max(_cal_amps)                 # renormalize so max = 1
_cal_correction = 1.0 / _cal_amps                          # correction multiplier at each calibration point

def _linear_func(x, m, b):
    return m * x + b

_fit_mask = _cal_freq_mhz < -50
_fit_popt, _ = optimize.curve_fit(_linear_func, _cal_freq_mhz[_fit_mask], _cal_correction[_fit_mask])
print(f"Amplitude-correction linear extrapolation (f < -50 MHz): "
      f"slope={_fit_popt[0]:.6f}, intercept={_fit_popt[1]:.6f}")

def get_correction_multiplier(input_freq_hz):
    """Frequency (Hz, scalar or array) -> multiplicative amplitude correction.
    The calibration was only measured for f < 0 MHz, but the diffraction
    response is symmetric about 0 MHz, so positive frequencies are folded
    onto the calibrated negative side (correction(+f) = correction(-f))
    before doing the same interpolation/extrapolation as the negative side:
    interpolate the measured curve for -50 MHz <= f <= 0, and linearly
    extrapolate the fit for f < -50 MHz."""
    input_freq_mhz = np.asarray(input_freq_hz, dtype=float) / 1e6
    folded_freq_mhz = -np.abs(input_freq_mhz)  # mirror positive side onto calibrated negative side
    correction = np.interp(folded_freq_mhz, _cal_freq_mhz, _cal_correction)
    correction = np.where(folded_freq_mhz < -50, _linear_func(folded_freq_mhz, *_fit_popt), correction)
    return correction

# -----------------------------------------------------------------------------
# PER-STEP, PER-TONE RATIOS AND AMPLITUDES
#
# Both are (TOTAL_BUFFERS, NUM_TONES) arrays: one row per composite buffer
# -- rows 0..NUM_STEPS-1 are the sweep steps (same order as chirp_offsets_hz),
# and the final row (index NUM_STEPS) is the trap buffer, evaluated at
# TRAP_OFFSET_HZ. Columns follow BASE_TONES_HZ's tone order.
#
#   TONE_RATIOS_PER_STEP[i, j]  -- relative duration weight of tone j within
#                                   step i's composite buffer. Only the
#                                   ratio between the 4 numbers in a row
#                                   matters; each row is re-normalized to
#                                   sum to 1 independently.
#   AMPLITUDE_PER_STEP[i, j]    -- serrodyne amplitude (0-1) of tone j
#                                   within step i, AFTER frequency
#                                   correction (see below) -- this is what
#                                   actually gets passed to serrodyne_tone.
#
# Ratios default to a fixed global split (edit TONE_RATIOS_PER_STEP directly
# to vary them per step). Amplitudes are derived from the calibration above:
# each tone's actual frequency at each step is looked up against
# get_correction_multiplier, multiplied by BASE_AMPLITUDE. If that would
# push any entry above 1 (full DAC scale), BASE_AMPLITUDE is scaled down
# automatically and the applied scale factor is printed.
# -----------------------------------------------------------------------------
DEFAULT_TONE_RATIOS = np.array([0.337, 0.167, 0.288, 0.208])
BASE_AMPLITUDE = 0.5   # nominal amplitude before frequency correction (0-1)

TONE_RATIOS_PER_STEP = np.tile(DEFAULT_TONE_RATIOS, (TOTAL_BUFFERS, 1))   # (TOTAL_BUFFERS, 4)

_all_offsets_hz = np.concatenate([chirp_offsets_hz, [TRAP_OFFSET_HZ]])          # (TOTAL_BUFFERS,)
frequencies_hz = _all_offsets_hz[:, None] + BASE_TONES_HZ[None, :]              # (TOTAL_BUFFERS, 4)
correction_multiplier = get_correction_multiplier(frequencies_hz)               # (TOTAL_BUFFERS, 4)

_max_needed = BASE_AMPLITUDE * correction_multiplier.max()
if _max_needed > 1.0:
    _scale = 1.0 / _max_needed
    print(f"BASE_AMPLITUDE={BASE_AMPLITUDE:.4f} x max correction "
          f"({correction_multiplier.max():.4f}) = {_max_needed:.4f} > 1.0 -- "
          f"auto-scaling BASE_AMPLITUDE down by {_scale:.4f} to stay within DAC full scale.")
    BASE_AMPLITUDE *= _scale

AMPLITUDE_PER_STEP = BASE_AMPLITUDE * correction_multiplier   # (TOTAL_BUFFERS, 4)

print(f"Per-tone frequencies at each step (Hz), shape {frequencies_hz.shape}:")
print(np.array2string(frequencies_hz, separator=', ', precision=0, suppress_small=True))
print(f"Correction multipliers, shape {correction_multiplier.shape}:")
print(np.array2string(correction_multiplier, separator=', ', precision=3, suppress_small=True))
print(f"Final BASE_AMPLITUDE = {BASE_AMPLITUDE:.4f}; "
      f"AMPLITUDE_PER_STEP range = [{AMPLITUDE_PER_STEP.min():.4f}, {AMPLITUDE_PER_STEP.max():.4f}]")

assert TONE_RATIOS_PER_STEP.shape == (TOTAL_BUFFERS, NUM_TONES), (
    f"TONE_RATIOS_PER_STEP must be shape ({TOTAL_BUFFERS}, {NUM_TONES}), "
    f"got {TONE_RATIOS_PER_STEP.shape}"
)
assert AMPLITUDE_PER_STEP.shape == (TOTAL_BUFFERS, NUM_TONES), (
    f"AMPLITUDE_PER_STEP must be shape ({TOTAL_BUFFERS}, {NUM_TONES}), "
    f"got {AMPLITUDE_PER_STEP.shape}"
)
assert np.all(TONE_RATIOS_PER_STEP.sum(axis=1) > 0), "Each row of TONE_RATIOS_PER_STEP must sum to > 0"
assert np.all(AMPLITUDE_PER_STEP >= 0) and np.all(AMPLITUDE_PER_STEP <= 1), (
    "AMPLITUDE_PER_STEP entries must be within [0, 1]"
)

TOTAL_SWEEP_S = 45 # 6e-3 


max_feasible_total_s = 0.95 * ENV_MAXLEN / ENV_SR
CYCLE_S = max_feasible_total_s / TOTAL_BUFFERS   # initial (nominal) composite-buffer duration

tone_fractions_per_step = TONE_RATIOS_PER_STEP / TONE_RATIOS_PER_STEP.sum(axis=1, keepdims=True)

STEP_HOLD_S = TOTAL_SWEEP_S / NUM_STEPS
STEP_HOLD_US = STEP_HOLD_S * 1e6

print(f"Nominal composite buffer cycle: {CYCLE_S*1e9:.2f} ns (may shrink to fit memory -- "
      f"see below), held via mode='periodic' for {STEP_HOLD_US:.2f} us per chirp step "
      f"({TOTAL_SWEEP_S*1e3:.3f} ms total sweep)")
print("Per-step tone ratios/amplitudes (step 0 shown as example, at the nominal cycle; "
      "edit TONE_RATIOS_PER_STEP directly, or BASE_AMPLITUDE, to change these):")
_preview_durations = tone_fractions_per_step[0] * CYCLE_S
for j in range(NUM_TONES):
    print(f"  tone {j} ({BASE_TONES_HZ[j]/1e6:+.1f} MHz offset): "
          f"step0 ratio {TONE_RATIOS_PER_STEP[0, j]:.3f}, amp {AMPLITUDE_PER_STEP[0, j]:.3f} "
          f"-> {_preview_durations[j]*1e9:.2f} ns")

def build_multitone_buffer(chirp_offset_hz, phase0, tone_durations_s_row, amplitudes_row):
    """One composite envelope: all NUM_TONES tones, each shifted by the same
    chirp_offset_hz, using this buffer's own per-tone durations and
    amplitudes (a single row of tone_durations_s / AMPLITUDE_PER_STEP),
    concatenated. Sample count per tone is chosen (within a small
    grid-aligned window) to minimize the phase residual at the
    mode="periodic" wraparound point."""
    i_pieces, q_pieces = [], []
    phase = phase0
    for base_tone, dur, amp in zip(BASE_TONES_HZ, tone_durations_s_row, amplitudes_row):
        f = chirp_offset_hz + base_tone
        n_best = best_n_samples(f, dur, ENV_SR, samps_per_clk)
        y, phase, n_samples = serrodyne_tone(f, dur, ENV_SR, amplitude=amp, phase0=phase,
                                              n_samples_override=n_best)
        pad = (-len(y)) % samps_per_clk
        if pad:
            y = np.concatenate([y, np.zeros(pad)])
        i_pieces.append(np.round(y * maxv).astype(np.int16))
        q_pieces.append(np.zeros(len(y), dtype=np.int16))
    return np.concatenate(i_pieces), np.concatenate(q_pieces), phase, [len(p) for p in i_pieces]

maxv = soccfg.get_maxv(GEN_CH)

def build_all_buffers(cycle_s):
    """Build every sweep-step buffer plus the trap buffer at composite-buffer
    duration cycle_s, using each row's own ratios/amplitudes. Returns the raw
    (pre-equalization) sweep buffers, the trap buffer, and the flat list of
    every individual tone-slice length (for the hardware floor check)."""
    durations_s = tone_fractions_per_step * cycle_s  # (TOTAL_BUFFERS, 4)

    idata_raw, qdata_raw = [], []
    all_piece_lens = []
    phase = 0.0
    for i, offset in enumerate(chirp_offsets_hz):
        idata, qdata, phase, piece_lens = build_multitone_buffer(
            offset, phase, durations_s[i], AMPLITUDE_PER_STEP[i]
        )
        idata_raw.append(idata)
        qdata_raw.append(qdata)
        all_piece_lens.extend(piece_lens)

    trap_idata_, trap_qdata_, _, trap_piece_lens_ = build_multitone_buffer(
        TRAP_OFFSET_HZ, phase, durations_s[NUM_STEPS], AMPLITUDE_PER_STEP[NUM_STEPS]
    )
    all_piece_lens.extend(trap_piece_lens_)
    return idata_raw, qdata_raw, trap_idata_, trap_qdata_, trap_piece_lens_, all_piece_lens

# -----------------------------------------------------------------------------
# MEMORY-FIT LOOP WITH AUTOMATIC BUFFER-LENGTH EQUALIZATION
#
# The tProc addresses consecutive "serr_i" envelope buffers by incrementing
# a fixed address step each time the sweep advances -- that only works if
# every sweep-step buffer occupies the same number of envelope samples. Since
# per-tone durations are individually snapped to the fabric-clock grid
# (best_n_samples) and per-step ratios can differ row to row, raw buffer
# lengths can vary step to step; they're padded up to the longest one so all
# NUM_STEPS buffers come out equal length.
#
# That padding can push the total over ENV_MAXLEN. Since CYCLE_S (how many
# times each step's buffer loops within its fixed STEP_HOLD_US dwell) has no
# effect on the sweep's physical timing, it's safe to shrink automatically
# and rebuild until sweep + trap fit in memory, rather than requiring manual
# retuning of NUM_STEPS or the ratio/amplitude arrays.
# -----------------------------------------------------------------------------
SHRINK_FACTOR = 0.98
MAX_FIT_ATTEMPTS = 60

cycle_s = CYCLE_S
for attempt in range(1, MAX_FIT_ATTEMPTS + 1):
    idata_list_raw, qdata_list_raw, trap_idata, trap_qdata, trap_piece_lens, all_piece_lens = \
        build_all_buffers(cycle_s)

    min_piece_len = min(all_piece_lens)
    if min_piece_len < 3 * samps_per_clk:
        raise ValueError(
            f"Smallest tone slice is only {min_piece_len} samples "
            f"({min_piece_len/samps_per_clk:.1f} fabric cycles) at cycle_s={cycle_s*1e9:.2f} ns "
            f"(attempt {attempt}) -- hardware needs >= 3. Shrinking the buffer further only makes "
            f"this worse, so this must be fixed by hand: adjust TONE_RATIOS_PER_STEP (avoid "
            f"near-zero ratios in any row), or reduce NUM_STEPS."
        )

    raw_lengths = [len(x) for x in idata_list_raw]
    samples_per_step = max(raw_lengths)
    samples_per_step += (-samples_per_step) % samps_per_clk  # round up to fabric-cycle grid
    total_samples = samples_per_step * NUM_STEPS
    total_with_trap = total_samples + len(trap_idata)

    if total_with_trap <= ENV_MAXLEN:
        break
    cycle_s *= SHRINK_FACTOR
else:
    raise ValueError(
        f"Could not fit sweep + trap buffers in {ENV_MAXLEN} samples after "
        f"{MAX_FIT_ATTEMPTS} shrink attempts. Reduce NUM_STEPS, or make "
        f"TONE_RATIOS_PER_STEP less extreme (less step-to-step variation)."
    )

if attempt > 1:
    print(f"Composite buffer cycle shrunk from {CYCLE_S*1e9:.2f} ns to {cycle_s*1e9:.2f} ns "
          f"over {attempt} attempts to fit envelope memory (sweep timing unaffected).")

print(f"Smallest individual tone slice across all steps (incl. trap): {min_piece_len} samples "
      f"({min_piece_len/samps_per_clk:.1f} fabric cycles)")
print(f"Raw per-step buffer lengths (pre-padding): {sorted(set(raw_lengths))}")

idata_list, qdata_list = [], []
for idata, qdata in zip(idata_list_raw, qdata_list_raw):
    pad = samples_per_step - len(idata)
    if pad:
        idata = np.concatenate([idata, np.zeros(pad, dtype=np.int16)])
        qdata = np.concatenate([qdata, np.zeros(pad, dtype=np.int16)])
    idata_list.append(idata)
    qdata_list.append(qdata)

assert all(len(x) == samples_per_step for x in idata_list), "Sweep buffers not equal length after padding"
assert samples_per_step % samps_per_clk == 0

print(f"Equalized per-step buffer length: {samples_per_step} samples "
      f"(shortest raw step was {min(raw_lengths)} samples, so up to "
      f"{samples_per_step - min(raw_lengths)} samples of trailing zero padding added)")
print(f"Total envelope samples (sweep): {total_samples} / {ENV_MAXLEN} available")
print(f"Trap buffer: {len(trap_idata)} samples (tone slices: {trap_piece_lens})")
print(f"Total envelope samples (sweep + trap): {total_with_trap} / {ENV_MAXLEN} available")

# -----------------------------------------------------------------------------
# 4. PROGRAM: sweep as before, then ONE final pulse on the concatenated trap
#    buffer with mode="periodic" -- hardware loops it forever on its own.
# -----------------------------------------------------------------------------
class RepeatedStepSerrodyneProgram(RAveragerProgram):
    def initialize(self):
        cfg = self.cfg
        res_ch = cfg["res_ch"]

        self.declare_gen(ch=res_ch, nqz=1)

        for i, (idata_step, qdata_step) in enumerate(zip(cfg["idata_list"], cfg["qdata_list"])):
            self.add_envelope(ch=res_ch, name=f"serr_{i}", idata=idata_step, qdata=qdata_step)

        self.add_envelope(ch=res_ch, name="trap_wfm", idata=cfg["trap_idata"], qdata=cfg["trap_qdata"])

        self.set_pulse_registers(
            ch=res_ch,
            style="arb",
            freq=0,
            phase=0,
            gain=cfg["gain"],
            waveform="serr_0",
            outsel="input",
            mode="periodic",
        )

        self.r_rp = self.ch_page(res_ch)
        self.r_addr = self.sreg(res_ch, "addr")
        self.addr_step = cfg["samples_per_step"] // self.soccfg["gens"][res_ch]["samps_per_clk"]

        self.synci(200)

    def body(self):
        res_ch = self.cfg["res_ch"]
        step_cycles = self.us2cycles(self.cfg["step_hold_us"])

        self.trigger(pins=[0])
        self.pulse(ch=res_ch, t='auto')
        self.sync_all(step_cycles)

    def update(self):
        self.mathi(self.r_rp, self.r_addr, self.r_addr, '+', self.addr_step)

    def make_program(self):
        """Standard RAveragerProgram sweep (expts x reps), with a single
        trapping pulse appended after the loop -- no loop construct needed
        for trapping, since mode="periodic" makes the hardware repeat it
        on its own once triggered."""
        p = self
        rcount = 13
        rii = 14
        rjj = 15

        p.initialize()
        p.regwi(0, rcount, 0)
        p.regwi(0, rii, self.cfg['expts'] - 1)
        p.label("LOOP_I")
        p.regwi(0, rjj, self.cfg['reps'] - 1)
        p.label("LOOP_J")
        p.body()
        p.mathi(0, rcount, rcount, "+", 1)
        p.memwi(0, rcount, self.COUNTER_ADDR)
        p.loopnz(0, rjj, 'LOOP_J')
        p.update()
        p.loopnz(0, rii, "LOOP_I")

        # --- trapping: one pulse, periodic buffer, then end(). The DAC keeps
        # cycling the 4 concatenated tones forever regardless of what the
        # tProc does after this -- including after it hits end() and halts.
        res_ch = self.cfg["res_ch"]
        p.set_pulse_registers(
            ch=res_ch, style="arb", freq=0, phase=0, gain=self.cfg["gain"],
            waveform="trap_wfm", outsel="input", mode="periodic",
        )
        p.trigger(pins=[0])
        p.pulse(ch=res_ch, t='auto')
        p.end()

# -----------------------------------------------------------------------------
# 5. EXECUTION
# -----------------------------------------------------------------------------
config = {
    "res_ch": GEN_CH,
    "reps": 1,
    "expts": NUM_STEPS,
    "idata_list": idata_list,
    "qdata_list": qdata_list,
    "trap_idata": trap_idata,
    "trap_qdata": trap_qdata,
    "samples_per_step": samples_per_step,
    "step_hold_us": STEP_HOLD_US,
    "gain": 32767,
}

prog = RepeatedStepSerrodyneProgram(soccfg, config)
# prog.run(soc) #, start_src="external"
prog.run(soc, start_src="external")
print(f"Running on hardware — {NUM_STEPS} sweep steps x {STEP_HOLD_US:.2f} us "
      f"= {NUM_STEPS*STEP_HOLD_US*1e-3:.3f} ms sweep ({CHIRP_OFFSET_START_HZ/1e6:.0f} -> "
      f"{CHIRP_OFFSET_STOP_HZ/1e6:.0f} MHz), then trapping at {TRAP_OFFSET_HZ/1e6:.0f} MHz "
      f"(4 tones, periodic, indefinitely until soc.reset_gens()).")

Generator 0: f_fabric=614.400 MHz, samps_per_clk=16, envelope sample rate=9.8304 GSPS
Envelope memory available: 65536 samples
Amplitude-correction linear extrapolation (f < -50 MHz): slope=-0.002534, intercept=1.004077
Per-tone frequencies at each step (Hz), shape (46, 4):
[[-3.e+08, -2.e+08, -2.e+08, -2.e+08],
 [-3.e+08, -2.e+08, -2.e+08, -1.e+08],
 [-3.e+08, -2.e+08, -2.e+08, -1.e+08],
 [-3.e+08, -2.e+08, -2.e+08, -1.e+08],
 [-3.e+08, -2.e+08, -1.e+08, -1.e+08],
 [-3.e+08, -2.e+08, -1.e+08, -1.e+08],
 [-3.e+08, -2.e+08, -1.e+08, -1.e+08],
 [-2.e+08, -2.e+08, -1.e+08, -1.e+08],
 [-2.e+08, -2.e+08, -1.e+08, -9.e+07],
 [-2.e+08, -2.e+08, -1.e+08, -8.e+07],
 [-2.e+08, -1.e+08, -1.e+08, -8.e+07],
 [-2.e+08, -1.e+08, -9.e+07, -7.e+07],
 [-2.e+08, -1.e+08, -9.e+07, -6.e+07],
 [-2.e+08, -1.e+08, -8.e+07, -5.e+07],
 [-2.e+08, -1.e+08, -7.e+07, -5.e+07],
 [-2.e+08, -1.e+08, -6.e+07, -4.e+07],
 [-2.e+08, -1.e+08, -5.e+07, -3.e+07],
 [-2.e+08, -9.e+07, -5.e+07, -2.e+07],
 [-2.e+08, -9.e+07, -4.

In [15]:
soc.reset_gens()
